# Final NBA Forecast Analysis

This notebook summarizes the completed NBA forecasting project, including chronological model evaluation, 2026-27 preseason record projections, 2027 postseason simulation probabilities, and final forecast conclusions.


## 1. Project Overview

The project uses historical NBA regular-season data from 2018-19 through 2025-26. The modeling dataset contains 9,509 regular-season games and 84 leakage-safe pregame features built from prior team performance only.

The 2025-26 season was held out as a completely unseen chronological evaluation season. After model selection, the final production model was retrained using all available games through 2025-26. The forecast target is the 2026-27 regular season and the 2027 playoffs.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#d9e1ec",
    "axes.labelcolor": "#121826",
    "axes.titlecolor": "#121826",
    "xtick.color": "#263246",
    "ytick.color": "#263246",
    "font.size": 10,
    "axes.grid": True,
    "grid.color": "#e6ebf2",
    "grid.linewidth": 0.8,
})

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

EVALUATION_PATH = DATA_DIR / "model_evaluation.csv"
HOLDOUT_PATH = DATA_DIR / "holdout_predictions.csv"
SEASON_PATH = DATA_DIR / "season_2026_27_predictions.csv"
STANDINGS_PATH = DATA_DIR / "2027_projected_standings.csv"
PLAYOFFS_PATH = DATA_DIR / "2027_playoff_probabilities.csv"
SUMMARY_PATH = DATA_DIR / "forecast_2026_27_summary.json"

for input_path in [
    EVALUATION_PATH,
    HOLDOUT_PATH,
    SEASON_PATH,
    STANDINGS_PATH,
    PLAYOFFS_PATH,
    SUMMARY_PATH,
]:
    if not input_path.exists():
        raise FileNotFoundError(f"Missing required input file: {input_path}")

PROJECT_CONTEXT = pd.DataFrame(
    [
        ["Historical seasons", "2018-19 through 2025-26"],
        ["Regular-season games", "9,509"],
        ["Pregame feature count", "84"],
        ["Holdout evaluation season", "2025-26"],
        ["Forecast target", "2026-27 regular season and 2027 playoffs"],
    ],
    columns=["Item", "Value"],
)

PROJECT_CONTEXT


## 2. Model Evaluation

The evaluation set is the full 2025-26 season. The table below compares the candidate models using accuracy, ROC AUC, Log Loss, and Brier Score.

Accuracy and ROC AUC are higher-is-better. Log Loss and Brier Score are lower-is-better, so they are charted separately instead of collapsed into a combined score.


In [ ]:
evaluation = pd.read_csv(EVALUATION_PATH)
required_eval_columns = ["MODEL", "ACCURACY", "ROC_AUC", "LOG_LOSS", "BRIER_SCORE"]
missing_eval_columns = sorted(set(required_eval_columns) - set(evaluation.columns))
if missing_eval_columns:
    raise ValueError(f"Evaluation file is missing columns: {missing_eval_columns}")

evaluation_display = evaluation.copy()
evaluation_display[required_eval_columns[1:]] = evaluation_display[required_eval_columns[1:]].round(4)
evaluation_display


In [ ]:
metric_specs = [
    ("ACCURACY", "Accuracy", True),
    ("ROC_AUC", "ROC AUC", True),
    ("LOG_LOSS", "Log Loss", False),
    ("BRIER_SCORE", "Brier Score", False),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, (metric, title, higher_is_better) in zip(axes, metric_specs):
    ordered = evaluation.sort_values(metric, ascending=not higher_is_better)
    colors = ["#1f5fbf" if model == "Logistic Regression" else "#9aa6b2" for model in ordered["MODEL"]]
    ax.barh(ordered["MODEL"], ordered[metric], color=colors)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel("Higher is better" if higher_is_better else "Lower is better")
    for index, value in enumerate(ordered[metric]):
        ax.text(value, index, f" {value:.3f}", va="center", fontsize=9)

fig.suptitle("Model Performance on the 2025-26 Holdout Season", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:

model_notes = []
for _, row in evaluation.iterrows():
    model = row["MODEL"]
    metrics = (
        f"Accuracy {row['ACCURACY']:.3f}, ROC AUC {row['ROC_AUC']:.3f}, "
        f"Log Loss {row['LOG_LOSS']:.3f}, Brier Score {row['BRIER_SCORE']:.3f}"
    )

    if model == "Logistic Regression":
        interpretation = (
            "selected because it produced the strongest probability-quality results, "
            "which matters most for downstream simulation."
        )
    elif model == "HistGradientBoosting":
        interpretation = (
            "had the highest accuracy, but its Log Loss and Brier Score were weaker "
            "than Logistic Regression."
        )
    elif model == "Random Forest":
        interpretation = (
            "was competitive on ROC AUC, but its probability-quality metrics trailed "
            "the selected model."
        )
    elif model == "Dummy Baseline":
        interpretation = (
            "sets the baseline by predicting the prior class distribution and confirms "
            "that the trained models add meaningful signal."
        )
    else:
        interpretation = "included for comparison."

    model_notes.append(f"- **{model}:** {metrics}; {interpretation}")

display(Markdown("### Model-by-model interpretation\n" + "\n".join(model_notes)))


In [ ]:
logistic_row = evaluation.loc[evaluation["MODEL"].eq("Logistic Regression")]
if logistic_row.empty:
    raise ValueError("Logistic Regression row not found in model evaluation file.")
logistic_row = logistic_row.iloc[0]

best_log_loss = evaluation.loc[evaluation["LOG_LOSS"].idxmin()]
best_brier = evaluation.loc[evaluation["BRIER_SCORE"].idxmin()]

display(Markdown(
    f"""
**Selected model:** Logistic Regression. It was selected primarily because it produced the best probability-quality metrics on the unseen 2025-26 holdout: Log Loss **{logistic_row['LOG_LOSS']:.3f}** and Brier Score **{logistic_row['BRIER_SCORE']:.3f}**. Its Accuracy was **{logistic_row['ACCURACY']:.3f}** and ROC AUC was **{logistic_row['ROC_AUC']:.3f}**.

Log Loss and Brier Score matter here because the predicted probabilities feed the Monte Carlo postseason simulator. A model that produces better-calibrated probabilities is more useful for simulation than a model selected only for classification accuracy.

Lowest Log Loss: **{best_log_loss['MODEL']}** ({best_log_loss['LOG_LOSS']:.3f}). Lowest Brier Score: **{best_brier['MODEL']}** ({best_brier['BRIER_SCORE']:.3f}).
"""
))


## 3. Holdout Prediction Analysis

The next section analyzes the final Logistic Regression probabilities on the unseen 2025-26 holdout season. The probability column is detected automatically from the holdout prediction file.


In [ ]:
holdout = pd.read_csv(HOLDOUT_PATH, dtype={"GAME_ID": str})
required_holdout_columns = ["GAME_ID", "GAME_DATE", "HOME_TEAM", "AWAY_TEAM", "HOME_WIN"]
missing_holdout_columns = sorted(set(required_holdout_columns) - set(holdout.columns))
if missing_holdout_columns:
    raise ValueError(f"Holdout prediction file is missing columns: {missing_holdout_columns}")

probability_columns = [column for column in holdout.columns if column.endswith("_HOME_WIN_PROB")]
logistic_probability_columns = [
    column for column in probability_columns
    if "LOGISTIC" in column.upper() and "REGRESSION" in column.upper()
]
if len(logistic_probability_columns) != 1:
    raise ValueError(
        "Expected exactly one Logistic Regression probability column. "
        f"Candidates found: {logistic_probability_columns}"
    )

lr_probability_column = logistic_probability_columns[0]
holdout["LR_HOME_WIN_PROB"] = holdout[lr_probability_column]
holdout["PREDICTED_HOME_WIN"] = (holdout["LR_HOME_WIN_PROB"] >= 0.50).astype(int)
holdout["THRESHOLD_CORRECT"] = holdout["PREDICTED_HOME_WIN"].eq(holdout["HOME_WIN"])
holdout["PREDICTED_WINNER"] = np.where(
    holdout["PREDICTED_HOME_WIN"].eq(1),
    holdout["HOME_TEAM"],
    holdout["AWAY_TEAM"],
)
holdout["WINNING_PROBABILITY"] = np.where(
    holdout["PREDICTED_HOME_WIN"].eq(1),
    holdout["LR_HOME_WIN_PROB"],
    1 - holdout["LR_HOME_WIN_PROB"],
)
holdout["DISTANCE_FROM_50"] = (holdout["LR_HOME_WIN_PROB"] - 0.50).abs()

holdout_summary = pd.DataFrame(
    [
        ["Logistic Regression probability column", lr_probability_column],
        ["Holdout games", f"{len(holdout):,}"],
        ["Average predicted home-win probability", f"{holdout['LR_HOME_WIN_PROB'].mean():.3f}"],
        ["Actual home-win rate", f"{holdout['HOME_WIN'].mean():.3f}"],
        ["Thresholded accuracy at 0.50", f"{holdout['THRESHOLD_CORRECT'].mean():.3f}"],
    ],
    columns=["Metric", "Value"],
)

holdout_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(holdout["LR_HOME_WIN_PROB"], bins=np.linspace(0, 1, 21), color="#1f5fbf", edgecolor="white")
ax.axvline(0.50, color="#c83e4d", linewidth=2, label="50% threshold")
ax.set_title("Distribution of Logistic Regression Home-Win Probabilities")
ax.set_xlabel("Predicted home-win probability")
ax.set_ylabel("Number of games")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
highest_confidence = holdout.sort_values(
    ["WINNING_PROBABILITY", "DISTANCE_FROM_50"],
    ascending=[False, False],
).head(10)[
    [
        "GAME_DATE",
        "AWAY_TEAM",
        "HOME_TEAM",
        "HOME_WIN",
        "LR_HOME_WIN_PROB",
        "PREDICTED_WINNER",
        "WINNING_PROBABILITY",
        "THRESHOLD_CORRECT",
    ]
]

closest_to_50 = holdout.sort_values(
    ["DISTANCE_FROM_50", "GAME_DATE", "GAME_ID"],
    ascending=[True, True, True],
).head(10)[
    [
        "GAME_DATE",
        "AWAY_TEAM",
        "HOME_TEAM",
        "HOME_WIN",
        "LR_HOME_WIN_PROB",
        "PREDICTED_WINNER",
        "WINNING_PROBABILITY",
        "THRESHOLD_CORRECT",
    ]
]

display(Markdown("### Highest-confidence holdout predictions"))
display(highest_confidence.round({"LR_HOME_WIN_PROB": 3, "WINNING_PROBABILITY": 3}))

display(Markdown("### Closest-to-50% holdout predictions"))
display(closest_to_50.round({"LR_HOME_WIN_PROB": 3, "WINNING_PROBABILITY": 3}))


In [ ]:
calibration_bins = np.linspace(0, 1, 11)
holdout["PROBABILITY_BIN"] = pd.cut(
    holdout["LR_HOME_WIN_PROB"],
    bins=calibration_bins,
    include_lowest=True,
)

calibration = (
    holdout.groupby("PROBABILITY_BIN", observed=True)
    .agg(
        average_predicted_probability=("LR_HOME_WIN_PROB", "mean"),
        actual_home_win_rate=("HOME_WIN", "mean"),
        number_of_games=("GAME_ID", "count"),
    )
    .reset_index()
)

calibration_display = calibration.copy()
calibration_display["PROBABILITY_BIN"] = calibration_display["PROBABILITY_BIN"].astype(str)
display(calibration_display.round(3))

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot([0, 1], [0, 1], color="#8a96a8", linestyle="--", label="Perfect calibration reference")
ax.scatter(
    calibration["average_predicted_probability"],
    calibration["actual_home_win_rate"],
    s=np.maximum(calibration["number_of_games"], 20),
    color="#1f5fbf",
    alpha=0.85,
)
for _, row in calibration.iterrows():
    ax.text(
        row["average_predicted_probability"],
        row["actual_home_win_rate"],
        f" {int(row['number_of_games'])}",
        va="center",
        fontsize=8,
    )
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Average predicted home-win probability")
ax.set_ylabel("Actual home-win rate")
ax.set_title("Calibration by Probability Bin")
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown(
    "The calibration plot compares average predicted probabilities with observed home-win rates by bin. "
    "The model is useful for probability-driven simulation, but the bins do not indicate perfect calibration."
))


## 4. 2026-27 Season Projections

These are model-implied preseason strength projections. They are not full simulations of the official 2026-27 schedule.


In [ ]:
season = pd.read_csv(SEASON_PATH)
required_season_columns = ["TEAM", "PROJECTED_WINS", "PROJECTED_LOSSES", "PROJECTED_WIN_PCT"]
missing_season_columns = sorted(set(required_season_columns) - set(season.columns))
if missing_season_columns:
    raise ValueError(f"Season projection file is missing columns: {missing_season_columns}")

season_sorted = season.sort_values(
    ["PROJECTED_WINS", "PROJECTED_WIN_PCT", "TEAM"],
    ascending=[False, False, True],
).reset_index(drop=True)

top_team_row = season_sorted.iloc[0]
top_5_records = season_sorted.head(5).copy()

display(season_sorted)
display(Markdown(
    f"Highest projected team: **{top_team_row['TEAM']}** with **{int(top_team_row['PROJECTED_WINS'])}** projected wins."
))
display(Markdown("### Top 5 projected records"))
display(top_5_records)


In [ ]:
plot_rows = season_sorted.sort_values("PROJECTED_WINS", ascending=True)
fig, ax = plt.subplots(figsize=(10, 10))
colors = ["#1f5fbf" if team == top_team_row["TEAM"] else "#6f7f93" for team in plot_rows["TEAM"]]
ax.barh(plot_rows["TEAM"], plot_rows["PROJECTED_WINS"], color=colors)
ax.set_title("Projected 2026-27 Wins by Team")
ax.set_xlabel("Projected wins")
ax.set_ylabel("Team")
for index, value in enumerate(plot_rows["PROJECTED_WINS"]):
    ax.text(value + 0.3, index, f"{int(value)}", va="center", fontsize=8)
ax.set_xlim(0, max(plot_rows["PROJECTED_WINS"]) + 6)
plt.tight_layout()
plt.show()


## 5. Conference Standings

The projected standings file contains seeds 1-10 for each conference. Seeds 1-6 are direct playoff positions, while seeds 7-10 feed the Play-In simulation.


In [ ]:
standings = pd.read_csv(STANDINGS_PATH)
required_standings_columns = [
    "CONFERENCE",
    "SEED",
    "TEAM",
    "PROJECTED_WINS",
    "PROJECTED_LOSSES",
    "PROJECTED_WIN_PCT",
]
missing_standings_columns = sorted(set(required_standings_columns) - set(standings.columns))
if missing_standings_columns:
    raise ValueError(f"Projected standings file is missing columns: {missing_standings_columns}")

for conference in ["Eastern", "Western"]:
    display(Markdown(f"### {conference} Conference projected seeds"))
    display(
        standings.loc[standings["CONFERENCE"].eq(conference), required_standings_columns]
        .sort_values("SEED")
        .reset_index(drop=True)
    )

conference_leaders = (
    standings.sort_values(["CONFERENCE", "SEED"])
    .groupby("CONFERENCE", as_index=False)
    .first()[["CONFERENCE", "TEAM", "PROJECTED_WINS", "PROJECTED_LOSSES", "PROJECTED_WIN_PCT"]]
)
display(Markdown("### Strongest projected team by conference"))
display(conference_leaders)


## 6. 2027 Postseason Simulation

The postseason simulation is conditional on the projected regular-season standings. Seeds 1-6 are treated as automatic playoff qualifiers, seeds 7-10 enter the Play-In simulation, and seeds 11-15 are outside the projected postseason field.

The round-advancement and championship probabilities come from 10,000 postseason Monte Carlo simulations. The `MAKE_PLAYOFFS_PROB` column should be interpreted under this fixed projected-standings setup, not as a full-season qualification probability.


In [ ]:
playoffs = pd.read_csv(PLAYOFFS_PATH)
required_playoff_columns = [
    "TEAM",
    "CONFERENCE",
    "PROJECTED_SEED",
    "MAKE_PLAYOFFS_PROB",
    "CONF_SEMIFINALS_PROB",
    "CONF_FINALS_PROB",
    "NBA_FINALS_PROB",
    "CHAMPIONSHIP_PROB",
]
missing_playoff_columns = sorted(set(required_playoff_columns) - set(playoffs.columns))
if missing_playoff_columns:
    raise ValueError(f"Playoff probability file is missing columns: {missing_playoff_columns}")

playoffs_sorted = playoffs.sort_values(
    ["CHAMPIONSHIP_PROB", "NBA_FINALS_PROB", "TEAM"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(playoffs_sorted)

championship_probability_sum = playoffs["CHAMPIONSHIP_PROB"].sum()
most_likely_east = playoffs.loc[playoffs["CONFERENCE"].eq("Eastern")].sort_values(
    ["NBA_FINALS_PROB", "CHAMPIONSHIP_PROB"],
    ascending=[False, False],
).iloc[0]
most_likely_west = playoffs.loc[playoffs["CONFERENCE"].eq("Western")].sort_values(
    ["NBA_FINALS_PROB", "CHAMPIONSHIP_PROB"],
    ascending=[False, False],
).iloc[0]
most_likely_champion = playoffs_sorted.iloc[0]
top_5_championship = playoffs_sorted.head(5)[
    ["TEAM", "CONFERENCE", "PROJECTED_SEED", "NBA_FINALS_PROB", "CHAMPIONSHIP_PROB"]
]

display(Markdown(
    f"Championship probabilities sum to **{championship_probability_sum:.4f}**, which is approximately 1.0."
))
display(Markdown(
    f"Most likely Eastern Conference champion: **{most_likely_east['TEAM']}**.  "
    f"Most likely Western Conference champion: **{most_likely_west['TEAM']}**.  "
    f"Most likely NBA champion: **{most_likely_champion['TEAM']}**."
))
display(Markdown("### Top 5 championship probabilities"))
display(top_5_championship)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 10))

finals_rows = playoffs.sort_values("NBA_FINALS_PROB", ascending=True)
axes[0].barh(finals_rows["TEAM"], finals_rows["NBA_FINALS_PROB"] * 100, color="#1f5fbf")
axes[0].set_title("NBA Finals Probability")
axes[0].set_xlabel("Probability (%)")
axes[0].set_ylabel("Team")

championship_rows = playoffs.sort_values("CHAMPIONSHIP_PROB", ascending=True)
axes[1].barh(championship_rows["TEAM"], championship_rows["CHAMPIONSHIP_PROB"] * 100, color="#1f8a5b")
axes[1].set_title("Championship Probability")
axes[1].set_xlabel("Probability (%)")
axes[1].set_ylabel("")

fig.suptitle("2027 Postseason Simulation Probabilities", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()


## 7. Final Forecast Summary

The summary file provides the headline forecast outputs used by the frontend dashboard.


In [ ]:
with open(SUMMARY_PATH, "r") as file:
    summary = json.load(file)

highest_wins = summary["highest_projected_regular_season_win_total"]
summary_table = pd.DataFrame(
    [
        ["Projected East #1 seed", summary["projected_eastern_conference_1_seed"]],
        ["Projected West #1 seed", summary["projected_western_conference_1_seed"]],
        ["Highest projected regular-season wins", f"{highest_wins['team']} - {highest_wins['projected_wins']} wins"],
        ["Projected Eastern champion", summary["most_likely_eastern_conference_champion"]],
        ["Projected Western champion", summary["most_likely_western_conference_champion"]],
        ["Projected NBA champion", summary["most_likely_nba_champion"]],
        ["Championship probability", f"{summary['championship_probability'] * 100:.2f}%"],
    ],
    columns=["Forecast item", "Value"],
)

summary_table


## 8. Model Improvement Over the Original Version

The final model improved over the previous best model and was evaluated using a stronger chronological holdout design. The previous model metrics below are the reference values from the earlier project version.


In [ ]:
previous_metrics = {
    "Accuracy": 0.615,
    "ROC AUC": 0.672,
    "Log Loss": 0.649,
}

final_metrics = {
    "Accuracy": float(logistic_row["ACCURACY"]),
    "ROC AUC": float(logistic_row["ROC_AUC"]),
    "Log Loss": float(logistic_row["LOG_LOSS"]),
    "Brier Score": float(logistic_row["BRIER_SCORE"]),
}

improvement_rows = []
for metric in ["Accuracy", "ROC AUC", "Log Loss", "Brier Score"]:
    previous_value = previous_metrics.get(metric)
    final_value = final_metrics[metric]

    if previous_value is None:
        improvement = np.nan
        note = "No previous reference"
    elif metric == "Log Loss":
        improvement = previous_value - final_value
        note = "Lower is better"
    else:
        improvement = final_value - previous_value
        note = "Higher is better"

    improvement_rows.append(
        {
            "Metric": metric,
            "Previous best": previous_value,
            "Final Logistic Regression": final_value,
            "Improvement": improvement,
            "Interpretation": note,
        }
    )

improvement_table = pd.DataFrame(improvement_rows)
display(improvement_table.round(4))
display(Markdown(
    "The final evaluation is stronger methodologically because the entire 2025-26 season was reserved as an unseen chronological holdout rather than using a random split."
))


## 9. Limitations

- Injuries are not directly modeled.
- Every offseason roster transaction is not fully modeled.
- Preseason team states are primarily derived from completed 2025-26 team performance.
- Regular-season projections are model-implied strength estimates, not official-schedule simulations.
- Postseason simulations are conditional on the projected standings.
- Predictions are probabilistic, not guarantees.
- Performance on the 2025-26 holdout does not guarantee identical future performance.


## 10. Final Conclusion

This completed project combines NBA data collection, data cleaning, leakage-safe feature engineering, chronological model validation, probability-based model selection, Monte Carlo postseason simulation, production model inference, scheduled-game prediction, and frontend visualization.

The final forecast is strongest as a transparent probability system: it does not claim certainty, but it turns historical team performance into evaluated game probabilities, projected standings, and postseason championship odds that can be inspected and updated as better inputs become available.
